# 🔍 AML Detection — Money Laundering Detection Pipeline
**Dataset:** SAML-D (Synthetic Anti-Money Laundering Dataset)

---
### Cara download dataset:
1. Buka https://www.kaggle.com/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml
2. Klik tombol **Download** (perlu login Kaggle)
3. Ekstrak file ZIP, lalu taruh file CSV-nya di folder yang sama dengan notebook ini
4. Ganti variabel `DATASET_PATH` di cell berikutnya sesuai nama file CSV kamu
---

## 0. Install Library yang Dibutuhkan

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn xgboost lightgbm imbalanced-learn matplotlib seaborn

## 1. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score,
    roc_auc_score, f1_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import RandomOverSampler
import joblib

# =============================================
# REPLACE THIS WITH YOUR CSV FILE PATH
DATASET_PATH = r"C:\Users\nelae\Downloads\pencucian_uang_updated (2)\pencucian uang\pencucian uang\SAML-D.csv"
# =============================================

df = pd.read_csv(DATASET_PATH)

print(f'✅ Dataset loaded successfully!')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

df.head()

## 2. Eksplorasi Data (EDA)

In [ ]:
# General dataset information
print('=== Dataset Information ===')
print(df.info())
print()

print('=== Column Names ===')
print(df.columns.tolist())

# Check missing values
missing = df.isnull().sum()

print()
print('=== Missing Values ===')
print(missing[missing > 0] if missing.any() else '✅ No missing values found')

In [ ]:
import os
import matplotlib.pyplot as plt

# Output folder sesuai folder GitHub project kamu
OUTPUT_DIR = r"C:\Users\nelae\Downloads\pencucian_uang_updated (1)\pencucian uang\pencucian uang"

# Pastikan folder ada
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Current working directory:")
print(os.getcwd())

print("\nPNG files will be saved to:")
print(OUTPUT_DIR)

In [ ]:
# Automatically detect the target column
TARGET_COL = 'Is_laundering'
print(f'Using target column: "{TARGET_COL}"')

counts = df[TARGET_COL].value_counts()
pct = df[TARGET_COL].value_counts(normalize=True) * 100

print(f'\nClass distribution:')
for k in counts.index:
    label = 'Normal' if k == 0 else 'Suspicious'
    print(f'  {label} ({k}): {counts[k]:,} ({pct[k]:.4f}%)')

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#5DCAA5', '#D85A30']

axes[0].bar(['Normal', 'Suspicious'], counts.values, color=colors, edgecolor='none')
axes[0].set_title('Class Distribution (Raw Count)', fontsize=13)
axes[0].set_ylabel('Number of Transactions')

axes[1].pie(
    pct.values,
    labels=['Normal', 'Suspicious'],
    colors=colors,
    autopct='%1.2f%%',
    startangle=90
)
axes[1].set_title('Class Proportion', fontsize=13)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('⚠️ The data is highly imbalanced! It needs to be handled before training.')

In [ ]:
# Amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Amount'], bins=50, color='#5DCAA5', edgecolor='none', alpha=0.8)
axes[0].set_title('Amount Distribution (All Transactions)', fontsize=13)
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')

df.boxplot(
    column='Amount',
    by=TARGET_COL,
    ax=axes[1],
    boxprops=dict(color='#534AB7'),
    medianprops=dict(color='#D85A30')
)

axes[1].set_title('Amount: Normal vs Suspicious', fontsize=13)
axes[1].set_xlabel('Class (0=Normal, 1=Suspicious)')

plt.suptitle('')
plt.tight_layout()
plt.savefig('amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Subsample Data

In [ ]:
# The dataset is too large (9.5 million rows) — perform subsampling first to avoid slow processing
# Take 200,000 normal transactions + all suspicious transactions

df_majority = df[df[TARGET_COL] == 0]
df_minority = df[df[TARGET_COL] == 1]

df_majority_sub = resample(
    df_majority,
    n_samples=200_000,
    random_state=42
)

df_sub = pd.concat([df_majority_sub, df_minority]).reset_index(drop=True)

print(f'✅ Subsampling completed!')
print(f'Size after subsampling: {df_sub.shape[0]:,} rows')

print(f'Class distribution after subsampling:')
print(df_sub[TARGET_COL].value_counts())

## 4. Preprocessing & Feature Engineering

In [ ]:
# Create a copy of df_sub
df_processed = df_sub.copy()

print('Available columns:')
for col in df_processed.columns:
    print(f'  - {col}: {df_processed[col].dtype}')

In [ ]:
# === FEATURE ENGINEERING ===

# 1. Extract features from time-related columns
time_candidates = [
    c for c in df_processed.columns
    if 'time' in c.lower() or 'date' in c.lower()
]
print(f'Time columns found: {time_candidates}')

for col in time_candidates:
    try:
        df_processed[col] = pd.to_datetime(df_processed[col])
        df_processed[f'{col}_hour'] = df_processed[col].dt.hour
        df_processed[f'{col}_dayofweek'] = df_processed[col].dt.dayofweek
        df_processed[f'{col}_month'] = df_processed[col].dt.month
        df_processed[f'{col}_is_weekend'] = (
            df_processed[col].dt.dayofweek >= 5
        ).astype(int)

        df_processed.drop(columns=[col], inplace=True)
        print(f'  ✅ Time features extracted from "{col}"')

    except Exception as e:
        print(f'  ⚠️ Failed to parse "{col}": {e}')

# 2. Currency mismatch flag
currency_cols = [
    c for c in df_processed.columns
    if 'currency' in c.lower()
]

if len(currency_cols) >= 2:
    df_processed['currency_mismatch'] = (
        df_processed[currency_cols[0]] != df_processed[currency_cols[1]]
    ).astype(int)

    print(f'\n✅ currency_mismatch feature created from {currency_cols}')
    print(
        f'   Transactions with currency mismatch: '
        f'{df_processed["currency_mismatch"].sum():,}'
    )

# 3. Aggregate features per sender
sender_candidates = [
    c for c in df_processed.columns
    if 'sender' in c.lower()
]

if sender_candidates:
    sender_col = sender_candidates[0]

    sender_stats = df_processed.groupby(sender_col)['Amount'].agg(
        sender_tx_count='count',
        sender_avg_amount='mean',
        sender_std_amount='std',
        sender_max_amount='max'
    ).reset_index()

    df_processed = df_processed.merge(sender_stats, on=sender_col, how='left')
    df_processed['sender_std_amount'] = df_processed['sender_std_amount'].fillna(0)

    df_processed['amount_vs_sender_avg'] = (
        df_processed['Amount'] / (df_processed['sender_avg_amount'] + 1)
    )

    print(f'\n✅ Sender aggregate features added')

print(f'\nTotal columns after feature engineering: {df_processed.shape[1]}')

In [ ]:
# === DROP DATA LEAKAGE ===
# Laundering_type contains the type of money laundering — this leaks the answer!
# In a real-world scenario, this column would not be available, so it must be removed.

leakage_cols = ['Laundering_type']
existing_leakage = [c for c in leakage_cols if c in df_processed.columns]

if existing_leakage:
    df_processed.drop(columns=existing_leakage, inplace=True)
    print(f'🚫 Dropped data leakage column(s): {existing_leakage}')
    print('✅ The model is no longer cheating!')

# === ENCODING CATEGORICAL COLUMNS ===
categorical_cols = df_processed.select_dtypes(
    include=['object', 'category']
).columns.tolist()

# Drop ID columns (account numbers) — not informative as features
id_candidates = [
    c for c in categorical_cols
    if any(x in c.lower() for x in ['account', 'id'])
]

if id_candidates:
    df_processed.drop(columns=id_candidates, inplace=True, errors='ignore')
    print(f'\nDropped ID columns: {id_candidates}')

# Update categorical column list
categorical_cols = df_processed.select_dtypes(
    include=['object', 'category']
).columns.tolist()

if TARGET_COL in categorical_cols:
    categorical_cols.remove(TARGET_COL)

print(f'\nColumns to be encoded: {categorical_cols}')

le = LabelEncoder()

for col in categorical_cols:
    df_processed[col] = df_processed[col].astype(str).fillna('UNKNOWN')
    df_processed[col] = le.fit_transform(df_processed[col])
    print(f'  ✅ Label encoded: {col}')

# Ensure the target column is numeric
df_processed[TARGET_COL] = pd.to_numeric(
    df_processed[TARGET_COL],
    errors='coerce'
).fillna(0).astype(int)

print(f'\n✅ Encoding completed. Shape: {df_processed.shape}')

In [ ]:
# === SPLIT FEATURES AND TARGET ===
X = df_processed.drop(columns=[TARGET_COL])
y = df_processed[TARGET_COL]

print(f'Features X: {X.shape}')
print(f'Target y: {y.shape}')

print(f'\nList of features used ({X.shape[1]} features):')
for idx, col in enumerate(X.columns):
    print(f'  [{idx}] {col}')

# Scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('\n✅ Scaling completed (StandardScaler)')

## 5. Split Data Training & Testing

In [ ]:
# Stratified split: 80% training / 20% testing
# Stratified means the proportion of suspicious and normal classes is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'✅ Split completed!')
print(f'  Train : {X_train.shape[0]:,} rows ({y_train.sum():,} suspicious)')
print(f'  Test  : {X_test.shape[0]:,} rows ({y_test.sum():,} suspicious)')

# Handle class imbalance using RandomOverSampler on the TRAINING DATA only
ros = RandomOverSampler(random_state=42, sampling_strategy=0.1)
X_train_sm, y_train_sm = ros.fit_resample(X_train, y_train)

print(f'\n✅ Oversampling completed!')
print(f'y_train distribution after oversampling:')
print(pd.Series(y_train_sm).value_counts())

## 6. Training Model

In [ ]:
# Model evaluation function
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    f1     = f1_score(y_te, y_pred)
    pr_auc = average_precision_score(y_te, y_proba)
    roc    = roc_auc_score(y_te, y_proba)

    print('\n' + '='*50)
    print(f'Model: {name}')
    print(f'  F1-Score : {f1:.4f}')
    print(f'  PR-AUC   : {pr_auc:.4f}')
    print(f'  ROC-AUC  : {roc:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_te, y_pred, target_names=['Normal', 'Suspicious']))
    return model, y_pred, y_proba, {'name': name, 'f1': f1, 'pr_auc': pr_auc, 'roc_auc': roc}

In [ ]:
# === MODEL 1: Random Forest ===
rf_model, rf_pred, rf_proba, rf_scores = evaluate_model(
    'Random Forest',
    RandomForestClassifier(n_estimators=100, max_depth=15,
                           class_weight='balanced', random_state=42, n_jobs=-1),
    X_train_sm, y_train_sm, X_test, y_test
)

In [ ]:
# === MODEL 2: XGBoost ===
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pw  = neg_count / pos_count

print(f'scale_pos_weight: {scale_pw:.1f}')

xgb_model, xgb_pred, xgb_proba, xgb_scores = evaluate_model(
    'XGBoost',
    XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                  scale_pos_weight=scale_pw, subsample=0.8, colsample_bytree=0.8,
                  eval_metric='aucpr', random_state=42, n_jobs=-1, verbosity=0),
    X_train_sm, y_train_sm, X_test, y_test
)

In [ ]:
# === MODEL 3: LightGBM ===
lgbm_model, lgbm_pred, lgbm_proba, lgbm_scores = evaluate_model(
    'LightGBM',
    LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                   class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1, verbose=-1),
    X_train_sm, y_train_sm, X_test, y_test
)

## 7. Visualisasi & Evaluasi

In [ ]:
# Comparison of all models
results    = [rf_scores, xgb_scores, lgbm_scores]
results_df = pd.DataFrame(results)
print('=== Model Comparison ===')
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x      = np.arange(len(results_df))
width  = 0.25
colors = ['#5DCAA5', '#534AB7', '#D85A30']

bars1 = ax.bar(x - width, results_df['f1'],      width, label='F1-Score', color=colors[0])
bars2 = ax.bar(x,          results_df['pr_auc'],  width, label='PR-AUC',  color=colors[1])
bars3 = ax.bar(x + width,  results_df['roc_auc'], width, label='ROC-AUC', color=colors[2])

ax.set_xticks(x)
ax.set_xticklabels(results_df['name'])
ax.set_ylim(0, 1.15)
ax.set_title('Model Metrics Comparison', fontsize=14)
ax.legend()
ax.set_ylabel('Score')

for bg in [bars1, bars2, bars3]:
    for b in bg:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix for all models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_info = [
    ('Random Forest', rf_pred),
    ('XGBoost',       xgb_pred),
    ('LightGBM',      lgbm_pred)
]

for ax, (name, pred) in zip(axes, models_info):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Suspicious'],
                yticklabels=['Normal', 'Suspicious'])
    ax.set_title(f'Confusion Matrix — {name}', fontsize=11)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Precision-Recall Curve
fig, ax = plt.subplots(figsize=(8, 6))

for name, proba, color in [
    ('Random Forest', rf_proba,   '#5DCAA5'),
    ('XGBoost',       xgb_proba,  '#534AB7'),
    ('LightGBM',      lgbm_proba, '#D85A30')
]:
    p, r, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(r, p, label=f'{name} (AP={ap:.3f})', color=color, linewidth=2)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve — all models
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))

for name, proba, color in [
    ('Random Forest', rf_proba,   '#5DCAA5'),
    ('XGBoost',       xgb_proba,  '#534AB7'),
    ('LightGBM',      lgbm_proba, '#D85A30')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score   = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_score:.3f})', color=color, linewidth=2)

# Diagonal line (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=14)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance XGBoost — pakai nama kolom asli
fi      = pd.Series(xgb_model.feature_importances_, index=X_scaled.columns)
fi_top20 = fi.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 7))
fi_top20.sort_values().plot.barh(ax=ax, color='#534AB7', edgecolor='none')
ax.set_title('Top 20 Feature Importance — XGBoost', fontsize=14)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 fitur terpenting:')
print(fi_top20.head(10).to_string())

## 8. XAI — Explainable AI dengan SHAP

SHAP (SHapley Additive exPlanations) digunakan untuk menjelaskan **mengapa** model memprediksi suatu transaksi sebagai mencurigakan. Ini penting untuk lomba karena model tidak hanya akurat, tapi juga bisa dijelaskan ke stakeholder.

In [ ]:
import sys
print(sys.executable)
!pip install shap

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap'])

In [ ]:
import shap

# Initialize JS for interactive visualization in the notebook
shap.initjs()

# Use LightGBM because it is the fastest option for SHAP TreeExplainer
# Take a sample of 2,000 rows from the test set to avoid long processing time
X_shap = X_test.sample(n=min(2000, len(X_test)), random_state=42)

print("Calculating SHAP values using TreeExplainer...")
explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer(X_shap)

print(f"✅ SHAP values calculated successfully!")
print(f"   SHAP values shape: {shap_values.values.shape}")

In [ ]:
# === PLOT 1: SHAP Summary Plot (Beeswarm) ===
# Shows which features are the most influential and the direction of their impact
# - RED color   = high feature value
# - BLUE color  = low feature value
# - Right side  = pushes the prediction toward "Suspicious"
# - Left side   = pushes the prediction toward "Normal"

print("=== SHAP Summary Plot ===")
print("Red = high feature value | Blue = low feature value")
print("Right = pushes toward Suspicious | Left = pushes toward Normal\n")

shap.summary_plot(shap_values, X_shap, show=False)
plt.title("SHAP Summary Plot — LightGBM (AML Detection)", fontsize=13)
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === PLOT 2: SHAP Bar Plot (Global Feature Importance) ===
# Bar chart version of SHAP — easier to read
# Shows the average |SHAP value| for each feature

shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Mean |SHAP|) — LightGBM", fontsize=13)
plt.tight_layout()
plt.savefig('shap_bar_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === PLOT 3: SHAP Waterfall Plot — 1 Suspicious Transaction ===
# Explains WHY one transaction is predicted as suspicious
# Each bar shows the contribution of each feature to the final prediction

# Find the index of suspicious transactions from X_shap
y_shap = y_test.loc[X_shap.index]
suspicious_idx = y_shap[y_shap == 1].index

if len(suspicious_idx) > 0:
    # Take the first suspicious transaction
    idx_in_shap = X_shap.index.get_loc(suspicious_idx[0])
    print(f"Explaining suspicious transaction (index: {suspicious_idx[0]})")
    shap.waterfall_plot(shap_values[idx_in_shap], show=False)
    plt.title("SHAP Waterfall — SUSPICIOUS Transaction", fontsize=12)
    plt.tight_layout()
    plt.savefig('shap_waterfall_suspicious.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ No suspicious transaction found in this sample, try increasing n_samples")

In [ ]:
# === PLOT 4: SHAP Waterfall Plot — 1 Normal Transaction ===
# For comparison: why is this transaction classified as normal?

y_shap = y_test.loc[X_shap.index]
normal_idx = y_shap[y_shap == 0].index

if len(normal_idx) > 0:
    idx_in_shap = X_shap.index.get_loc(normal_idx[0])
    print(f"Explaining normal transaction (index: {normal_idx[0]})")
    shap.waterfall_plot(shap_values[idx_in_shap], show=False)
    plt.title("SHAP Waterfall — NORMAL Transaction", fontsize=12)
    plt.tight_layout()
    plt.savefig('shap_waterfall_normal.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# === PLOT 5: SHAP Dependence Plot — Most Important Feature ===
# Shows the relationship between the feature value and its SHAP value
# Scattered points indicate interaction with other features

# Take the top 1 feature based on mean |SHAP|
top_feature = X_shap.columns[shap_values.values[:, :].mean(axis=0).argsort()[::-1][0]]
print(f"Dependence plot for the most important feature: {top_feature}")

shap.dependence_plot(top_feature, shap_values.values, X_shap, show=False)
plt.title(f"SHAP Dependence Plot — {top_feature}", fontsize=13)
plt.tight_layout()
plt.savefig('shap_dependence_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ All SHAP plots completed!")
print("Saved files: shap_summary_plot.png, shap_bar_plot.png,")
print("             shap_waterfall_suspicious.png, shap_waterfall_normal.png,")
print("             shap_dependence_plot.png")

## 8. Simpan Model Terbaik

In [ ]:
# Select the model with the highest PR-AUC
best       = max([rf_scores, xgb_scores, lgbm_scores], key=lambda x: x['pr_auc'])
model_map  = {'Random Forest': rf_model, 'XGBoost': xgb_model, 'LightGBM': lgbm_model}
best_model = model_map[best['name']]

print(f'Best model: {best["name"]} (PR-AUC = {best["pr_auc"]:.4f})')

joblib.dump(best_model, 'aml_best_model.pkl')
joblib.dump(scaler,     'aml_scaler.pkl')

print('✅ Model saved to: aml_best_model.pkl')
print('✅ Scaler saved to: aml_scaler.pkl')

## 9. Contoh Prediksi Transaksi Baru

In [ ]:
# Prediction demo using one data point from the test set
def predict_transaction(data_row):
    pred  = best_model.predict(data_row.values.reshape(1, -1))[0]
    proba = best_model.predict_proba(data_row.values.reshape(1, -1))[0][1]

    print(f'Prediction : {"⚠️ SUSPICIOUS" if pred == 1 else "✅ NORMAL"}')
    print(f'Suspicious probability: {proba:.4f} ({proba*100:.2f}%)')

    return pred, proba

sample = X_test.iloc[0]

print(f'Actual label: {y_test.iloc[0]} ({"Suspicious" if y_test.iloc[0] == 1 else "Normal"})')
predict_transaction(sample)

## 10. Inference Benchmark & Computational Performance

Mengukur efisiensi komputasi model LightGBM untuk menilai kelayakan **Edge AI deployment**.

| Metrik | Keterangan |
|---|---|
| Model Size on Disk | Ukuran file `.pkl` (MB) |
| Memory Footprint | **Delta RSS** saat model dimuat — bukan seluruh proses Python |
| Inference Latency | Mean / P95 / P99 per satu transaksi (ms) |
| Throughput (TPS) | Prediksi per detik, mode batch |
| CPU per-core (%) | Rata-rata CPU **dinormalisasi per logical core** — valid di Windows multi-core |
| Inference Memory Delta | Tambahan RSS akibat inferensi saja (peak − baseline) |

In [ ]:
import sys, os, time, json, threading, warnings, gc
import numpy as np
import psutil
import joblib

warnings.filterwarnings('ignore')

N_LOGICAL_CORES = psutil.cpu_count(logical=True) or 1
print(f'✅ Library siap | Logical cores terdeteksi: {N_LOGICAL_CORES}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# HELPER: ResourceMonitor + make_data
# ═══════════════════════════════════════════════════════════

class ResourceMonitor:
    """
    FIX WINDOWS MULTI-CORE:
      psutil.cpu_percent() tanpa interval = nilai kumulatif semua core.
      Contoh: 8 core x 79% = 631% — perlu dibagi N_LOGICAL_CORES.
      avg_cpu_per_core = raw / N_LOGICAL_CORES → nilai per-core.
    """
    def __init__(self, interval=0.1):
        self.interval    = interval
        self.cpu_raw     = []
        self.mem_samples = []
        self._running    = False
        self._thread     = None
        self._proc       = psutil.Process(os.getpid())
        # Init delta 2x agar sample pertama tidak 0 palsu
        self._proc.cpu_percent(interval=None)
        time.sleep(0.2)
        self._proc.cpu_percent(interval=None)

    def start(self):
        self._running = True
        self._thread  = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=5)

    def _poll(self):
        while self._running:
            self.cpu_raw.append(self._proc.cpu_percent(interval=None))
            self.mem_samples.append(self._proc.memory_info().rss / (1024**2))
            time.sleep(self.interval)

    @property
    def avg_cpu_per_core(self):
        """CPU rata-rata per logical core (%) — nilai yang benar untuk paper."""
        valid = [x for x in self.cpu_raw if x > 0]
        if not valid:
            return 0.0
        return round((sum(valid) / len(valid)) / N_LOGICAL_CORES, 2)

    @property
    def avg_cpu_total(self):
        """Total CPU semua core (%) — referensi raw psutil."""
        valid = [x for x in self.cpu_raw if x > 0]
        return round(sum(valid) / len(valid), 2) if valid else 0.0

    @property
    def peak_mem_mb(self):
        """Peak RSS selama monitoring (MB)."""
        return round(max(self.mem_samples), 2) if self.mem_samples else 0.0

    @property
    def n_valid(self):
        """Jumlah sample CPU yang valid (> 0)."""
        return len([x for x in self.cpu_raw if x > 0])


def make_data(n):
    rng = np.random.default_rng(seed=42)
    return np.column_stack([
        rng.integers(1000,9999,size=n), rng.integers(1000,9999,size=n),
        rng.exponential(5000,size=n),   rng.integers(0,10,size=n),
        rng.integers(0,10,size=n),      rng.integers(0,50,size=n),
        rng.integers(0,50,size=n),      rng.integers(0,5,size=n),
        rng.integers(0,24,size=n),      rng.integers(0,7,size=n),
        rng.integers(1,13,size=n),      rng.integers(0,2,size=n),
        rng.integers(0,24,size=n),      rng.integers(0,7,size=n),
        rng.integers(1,13,size=n),      rng.integers(0,2,size=n),
        rng.integers(0,2,size=n),       rng.integers(1,200,size=n),
        rng.exponential(4000,size=n),   rng.exponential(1500,size=n),
        rng.exponential(20000,size=n),  rng.normal(1.0,0.5,size=n),
    ]).astype(np.float64)


print(f'✅ ResourceMonitor & make_data terdefinisi')
print(f'   Logical cores: {N_LOGICAL_CORES} | Properties: peak_mem_mb, avg_cpu_per_core, avg_cpu_total, n_valid')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 10.1  MODEL SIZE ON DISK  +  MEMORY FOOTPRINT
# ═══════════════════════════════════════════════════════════

MODEL_PATH  = 'aml_best_model.pkl'
SCALER_PATH = 'aml_scaler.pkl'

model_size_mb  = os.path.getsize(MODEL_PATH)  / (1024**2)
scaler_size_mb = os.path.getsize(SCALER_PATH) / (1024**2)

print('━'*55)
print('  MODEL SIZE ON DISK')
print('━'*55)
print(f'  aml_best_model.pkl : {model_size_mb:.3f} MB')
print(f'  aml_scaler.pkl     : {scaler_size_mb:.3f} MB')
print(f'  Total              : {model_size_mb+scaler_size_mb:.3f} MB')

# Memory footprint = delta RSS, bukan seluruh proses
proc = psutil.Process(os.getpid())
gc.collect(); time.sleep(0.2)
rss_before = proc.memory_info().rss / (1024**2)
_m = joblib.load(MODEL_PATH)
_s = joblib.load(SCALER_PATH)
time.sleep(0.2)
rss_after     = proc.memory_info().rss / (1024**2)
mem_footprint = max(rss_after - rss_before, 0)

print()
print('━'*55)
print('  MEMORY FOOTPRINT (delta RSS saat load)')
print('━'*55)
print(f'  RSS sebelum load   : {rss_before:.2f} MB')
print(f'  RSS sesudah load   : {rss_after:.2f} MB')
print(f'  Footprint model    : {mem_footprint:.2f} MB  ← tambahan akibat model')
del _m, _s

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10.2  INFERENCE LATENCY  (warm-up 200 + ukur 1000 sampel)
# ═══════════════════════════════════════════════════════════

N_WARMUP  = 200
N_MEASURE = 1000
data      = make_data(N_WARMUP + N_MEASURE)

print(f'  Warm-up {N_WARMUP} inferensi...')
for i in range(N_WARMUP):
    best_model.predict(scaler.transform(data[i].reshape(1,-1)))
print('  ✓ Warm-up selesai')

lats = []
for i in range(N_WARMUP, N_WARMUP + N_MEASURE):
    row = data[i].reshape(1,-1)
    t0  = time.perf_counter()
    best_model.predict(scaler.transform(row))
    lats.append((time.perf_counter()-t0)*1000)

a = np.array(lats)
lat_results = {
    'n_samples' : N_MEASURE,
    'mean_ms'   : round(float(a.mean()), 4),
    'median_ms' : round(float(np.median(a)), 4),
    'p95_ms'    : round(float(np.percentile(a,95)), 4),
    'p99_ms'    : round(float(np.percentile(a,99)), 4),
    'min_ms'    : round(float(a.min()), 4),
    'max_ms'    : round(float(a.max()), 4),
    'std_ms'    : round(float(a.std()), 4),
}

print()
print('━'*55)
print('  INFERENCE LATENCY (ms)')
print('━'*55)
print(f'  Mean       : {lat_results["mean_ms"]:.4f} ms')
print(f'  Median     : {lat_results["median_ms"]:.4f} ms')
print(f'  P95        : {lat_results["p95_ms"]:.4f} ms  ← target SLA')
print(f'  P99        : {lat_results["p99_ms"]:.4f} ms  ← batas atas SLA')
print(f'  Min / Max  : {lat_results["min_ms"]:.4f} / {lat_results["max_ms"]:.4f} ms')
print(f'  Std Dev    : {lat_results["std_ms"]:.4f} ms')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10.3  THROUGHPUT BATCH (5.000 sampel)
# ═══════════════════════════════════════════════════════════

N_BATCH = 5000
bd = make_data(N_BATCH)
bs = scaler.transform(bd)

t0    = time.perf_counter()
preds = best_model.predict(bs)
t1    = time.perf_counter()

tput_results = {
    'n_samples'      : N_BATCH,
    'elapsed_s'      : round(t1-t0, 6),
    'tps'            : round(N_BATCH/(t1-t0), 2),
    'fraud_detected' : int(preds.sum()),
    'fraud_rate_pct' : round(float(preds.mean())*100, 2),
}

print('━'*55)
print('  TRANSACTION THROUGHPUT (Batch 5.000 sampel)')
print('━'*55)
print(f'  Elapsed time     : {tput_results["elapsed_s"]:.4f} s')
print(f'  Throughput (TPS) : {tput_results["tps"]:,.1f} transaksi/detik')
print(f'  Fraud terdeteksi : {tput_results["fraud_detected"]:,} ({tput_results["fraud_rate_pct"]}%)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10.4  STRESS TEST + CPU & MEMORY
#
# FIX 1 – CPU Windows multi-core:
#   psutil melaporkan total semua core (mis. 8 core x 79% = 631%).
#   Dinormalisasi dengan membagi N_LOGICAL_CORES → nilai per-core.
#
# FIX 2 – Memory yang relevan:
#   inference_mem_delta = peak_RSS_selama_loop - baseline_RSS
#   bukan peak_RSS mentah (yang mencerminkan seluruh proses).
#
# Loop ≥ 5 detik agar monitor merekam ≥ 40 sample CPU valid.
# ═══════════════════════════════════════════════════════════

MIN_DURATION = 5.0   # detik
BATCH_SIZE   = 5000

sd   = make_data(BATCH_SIZE)
ss   = scaler.transform(sd)
proc = psutil.Process(os.getpid())

gc.collect(); time.sleep(0.3)
baseline_mb = proc.memory_info().rss / (1024**2)

print(f'  Menjalankan loop inferensi ≥ {MIN_DURATION} detik...')
monitor = ResourceMonitor(interval=0.1)
monitor.start()

total_pred = 0
t_start    = time.perf_counter()
while (time.perf_counter() - t_start) < MIN_DURATION:
    total_pred += len(best_model.predict(ss))
elapsed_stress = time.perf_counter() - t_start
monitor.stop()

peak_mb  = monitor.peak_mem_mb
delta_mb = round(max(peak_mb - baseline_mb, 0), 2)

stress_results = {
    'total_predictions'      : total_pred,
    'elapsed_s'              : round(elapsed_stress, 3),
    'tps'                    : round(total_pred/elapsed_stress, 2),
    'avg_cpu_per_core_pct'   : monitor.avg_cpu_per_core,
    'avg_cpu_total_pct'      : monitor.avg_cpu_total,
    'n_logical_cores'        : N_LOGICAL_CORES,
    'baseline_rss_mb'        : round(baseline_mb, 2),
    'peak_rss_mb'            : peak_mb,
    'inference_mem_delta_mb' : delta_mb,
    'n_valid_cpu_samples'    : monitor.n_valid,
    'n_total_cpu_samples'    : len(monitor.cpu_raw),
}

print()
print('━'*55)
print('  STRESS TEST — CPU & MEMORY')
print('━'*55)
print(f'  Total prediksi            : {total_pred:,}')
print(f'  Elapsed                   : {elapsed_stress:.3f} s')
print(f'  Throughput (TPS)          : {total_pred/elapsed_stress:,.1f}')
print(f'  Avg CPU per-core (FIX)    : {monitor.avg_cpu_per_core:.2f} %  ← normalized/{N_LOGICAL_CORES} core')
print(f'  Avg CPU total raw         : {monitor.avg_cpu_total:.2f} %  ← referensi psutil')
print(f'  Baseline RSS              : {baseline_mb:.2f} MB')
print(f'  Peak RSS selama loop      : {peak_mb:.2f} MB')
print(f'  Inference memory delta    : {delta_mb:.2f} MB  ← akibat inferensi saja')
print(f'  CPU samples (valid/total) : {monitor.n_valid}/{len(monitor.cpu_raw)}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10.5  EDGE AI SUITABILITY ASSESSMENT
# ═══════════════════════════════════════════════════════════

checks = [
    ('Model Size < 10 MB',       model_size_mb < 10,                         f'{model_size_mb:.3f} MB'),
    ('Memory Footprint < 50 MB', mem_footprint < 50,                         f'{mem_footprint:.2f} MB'),
    ('Mean Latency < 5 ms',      lat_results['mean_ms'] < 5,                 f"{lat_results['mean_ms']:.4f} ms"),
    ('P99 Latency < 10 ms',      lat_results['p99_ms'] < 10,                 f"{lat_results['p99_ms']:.4f} ms"),
    ('Throughput > 1.000 TPS',   tput_results['tps'] > 1000,                 f"{tput_results['tps']:,.0f} TPS"),
    ('Avg CPU per-core < 80%',   stress_results['avg_cpu_per_core_pct']<80,  f"{stress_results['avg_cpu_per_core_pct']:.1f}%"),
]

print('━'*55)
print('  EDGE AI SUITABILITY ASSESSMENT')
print('━'*55)
for label, ok, val in checks:
    print(f'  {"✅" if ok else "❌"}  {label:<30} → {val}')

all_pass = all(ok for _,ok,_ in checks)
print()
print('  ✅ Model LAYAK untuk Edge AI deployment.' if all_pass
      else '  ⚠️  Beberapa kriteria edge belum terpenuhi.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10.6  SIMPAN KE JSON
# ═══════════════════════════════════════════════════════════

import platform
vm = psutil.virtual_memory()

benchmark_results = {
    'project'      : 'AML Fraud Detection — LightGBM Inference Benchmark',
    'system_info'  : {
        'platform'           : platform.system(),
        'cpu_logical_cores'  : psutil.cpu_count(logical=True),
        'cpu_physical_cores' : psutil.cpu_count(logical=False),
        'ram_total_gb'       : round(vm.total/1024**3, 2),
        'python_version'     : platform.python_version(),
    },
    'model_size_mb'         : round(model_size_mb, 3),
    'memory_footprint_mb'   : round(mem_footprint, 2),
    'latency'               : lat_results,
    'throughput_batch'      : tput_results,
    'stress_cpu_memory'     : stress_results,
    'edge_suitability'      : all_pass,
}

OUT = 'inference_metrics_updated.json'
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(benchmark_results, f, indent=2, ensure_ascii=False)

print(f'✅ Hasil disimpan ke: {OUT}')
print(json.dumps(benchmark_results, indent=2))

### 📊 Panduan Interpretasi

| Metrik | Threshold | Catatan |
|---|---|---|
| Mean / P99 Latency | < 5 ms / < 10 ms | Makin kecil = deteksi sebelum dana berpindah |
| Memory Footprint | < 50 MB | **Delta RSS** — bukan RSS total proses |
| Model Size | < 10 MB | Ringan untuk edge node & CI/CD |
| TPS | > 1.000/dtk | Mampu handle volume transaksi bank |
| CPU per-core | < 80% | Raw psutil dibagi jumlah logical core |

> **Catatan CPU di Windows:**  
> `psutil.cpu_percent()` melaporkan total semua core (mis. 8 core × 79% = **631%**).  
> Nilai yang benar untuk paper = **raw ÷ N_logical_cores** (mis. 631 ÷ 8 = **~79% per-core**).  
> Ini adalah perilaku normal psutil di Windows, bukan error.